# 1. Attention Mechanism

Attention allows models to focus on relevant parts of the input when making predictions.
It's the core innovation behind transformers!


In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np


## What is Attention?

Attention computes how much each element in a sequence should "attend to" (focus on) each other element.

The key idea: **Not all relationships are equally important!**


In [ ]:
# Simple example: computing attention scores
# Imagine we have 3 words: ["cat", "sat", "mat"]
# We want to know how much "cat" attends to "sat" and "mat"

# Represent words as vectors (embeddings)
# word_embeddings represents the encoded words in batch
word_embeddings = torch.tensor([
    [1.0, 2.0],  # "cat"
    [0.5, 1.5],  # "sat"
    [1.5, 0.5],  # "mat"
])

# Query: what we're looking for (the word "cat")
# Define the query to look the properties of "cat"
query = word_embeddings[0]  # "cat"

# Keys: what we're comparing against (all words)
# Keys represent what data are as characteristics or properties
keys = word_embeddings

# Compute attention scores: dot product between query and each key
# matmul (matrix multiplication) also can be used as "@" operation
# Remember: to make matrix multiplication possible, the query (1,2) need to be multiply with
#   transposed of keys 
#       (word_embeddings (3,2),
#        transposed word_embeddings (2,3))
# Therefore, multiplication of attention_scores (1,3)
attention_scores = torch.matmul(query, keys.T)

print("Word embeddings:")
print(word_embeddings)
print(f"\nQuery (cat): {query}")
print(f"\nAttention scores:")

# Position of word to be dot product to attention_score as same as word_embeddings
print(f"  cat -> cat: {attention_scores[0]:.2f}")
print(f"  cat -> sat: {attention_scores[1]:.2f}")
print(f"  cat -> mat: {attention_scores[2]:.2f}")

# Normalize with softmax to get attention weights
# Remember: softmax() is different from activation_fx() e.g sigmoid
#   activation_fx() -> used to data position in graph within hidden layer in non-linear gradient
#   softmax() -> used to get the probability in output layer to obtain the most possible results from predictions
#             -> the sum total of all outputs (logits) is 1
#             -> the probability calculated through 'key' index -> referring to the innermost dimension (dim=-1)
attention_weights = F.softmax(attention_scores, dim=-1)
print(f"\nAttention weights (after softmax):")
print(f"  cat -> cat: {attention_weights[0]:.3f}")
print(f"  cat -> sat: {attention_weights[1]:.3f}")
print(f"  cat -> mat: {attention_weights[2]:.3f}")
print(f"\nSum: {attention_weights.sum():.3f} (should be 1.0)")


Word embeddings:
tensor([[1.0000, 2.0000],
        [0.5000, 1.5000],
        [1.5000, 0.5000]])

Query (cat): tensor([1., 2.])

Attention scores:
  cat -> cat: 5.00
  cat -> sat: 3.50
  cat -> mat: 2.50

Attention weights (after softmax):
  cat -> cat: 0.766
  cat -> sat: 0.171
  cat -> mat: 0.063

Sum: 1.000 (should be 1.0)


## Attention Formula

Attention(Q, K, V) = softmax(QK^T / √d_k) × V

Where:
- **Q** (Query): What we're looking for
- **K** (Key): What we're comparing against
- **V** (Value): The actual information we extract
- **d_k**: Dimension of keys (for scaling)
§

In [ ]:
# Implementing scaled dot-product attention
def scaled_dot_product_attention(Q, K, V):
    """
    Compute scaled dot-product attention
    
    Args:
        Q: Query tensor [batch_size, seq_len, d_k]
        K: Key tensor [batch_size, seq_len, d_k]
        V: Value tensor [batch_size, seq_len, d_v]

        where
            batch_size: number of words
            seq_len: number of words' token
            d_k: dimension of keys (for scaling)
    
    Returns:
        Output tensor and attention weights
    """
    # Extracting dimension of keys (d_k)
    #   Taken from shape of Q in innermost dimension (-1)
    #   Remember: Query tensor shape dimension of [batch_size, seq_len, d_k]
    d_k = Q.size(-1)
    
    # Formula of attention_score = (QK^T/sqrt(d_k))V
    # Calculation involved:
    #   - obtain attention score (first and second: (QK^T/sqrt(d_k)))
    #   - obtain attention weight (third: softmax())
    #   - obtain output (fourth: (QK^T/sqrt(d_k))V)

    # First: Compute attention scores: QK^T
    #   Multiplying one query into all others keys
    #   Dimension
    #       Q (Query): (batch_size, seq_len, d_k)
    #       K (Key): (batch_size, seq_len, d_k)
    #       K^T (Key - Transposed): (batch_size, d_k, seq_len)
    scores = torch.matmul(Q, K.transpose(-2, -1)) # <-- swap dimension '-2' and '-1'
    
    # Second: Compute attention scores: QK^T/sqrt(d_k)
    # Scale by sqrt(d_k)
    scores = scores / np.sqrt(d_k)
    
    # Third: Apply softmax to get attention weights
    # to get probability to sum all logits to 1
    #   Execute the softmax() in the innermost layer (-1) of attention_score
    attention_weights = F.softmax(scores, dim=-1)
    
    # Fourth: Compute attention scores: (QK^T/sqrt(d_k))V
    # Apply weights to values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Example usage
batch_size, seq_len, d_k = 1, 4, 8

Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"Input shape - \nQ (Query): {Q.shape}, \nK (Key): {K.shape}, \nV (Value): {V.shape}")
print(f"Tranposed K: {K.transpose(-2,-1).shape}")
print(f"\nAttention weights (matmul(attention_score, K^T)) shape: {attn_weights.shape}")
print(f"\nAttention weights (first sequence):")
print(attn_weights[0])
print(f"Each row sums to 1: {attn_weights[0].sum(dim=-1)}")

print(f"\nOutput shape: {output.shape}")


Input shape - 
Q (Query): torch.Size([1, 4, 8]), 
K (Key): torch.Size([1, 4, 8]), 
V (Value): torch.Size([1, 4, 8])
Tranposed K: torch.Size([1, 8, 4])

Attention weights (matmul(attenstion_score, K^T)) shape: torch.Size([1, 4, 4])

Attention weights (first sequence):
tensor([[0.4334, 0.3247, 0.0582, 0.1836],
        [0.3494, 0.2410, 0.2411, 0.1685],
        [0.1078, 0.1593, 0.1248, 0.6081],
        [0.4144, 0.2423, 0.0881, 0.2552]])
Each row sums to 1: tensor([1.0000, 1.0000, 1.0000, 1.0000])

Output shape: torch.Size([1, 4, 8])
